In [1]:
# === IMPORT LIBRARIES ===
import os
import sys
import subprocess
import ctypes

# === CUDA SYSTEM BOOT FIX ===

# Force-inject CUDA library to RAM before ANY module imports!
try:
    ctypes.CDLL("/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib/libnvJitLink.so.13")
except Exception:
    pass

# SYSTEM HOTFIX: Inject absolute path to CUDA 13.0 linker libraries
# This guarantees that bitsandbytes and 4-bit quantization load flawlessly on this server!
cuda_link_path = "/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":" + cuda_link_path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# === FILEPATH SETUP ===

# 1. Inject Codebase into Python Path (Wipe cache first for Jupyter safety)
import sys
for key in list(sys.modules.keys()):
    if key.startswith("src"):
        del sys.modules[key]

codebase_path = "/home/emmy/emmy/mlbio/hw4"
if codebase_path not in sys.path:
    sys.path.insert(0, codebase_path)
print(f"✅ Codebase mounted at: {codebase_path}")

# 2. Configure Global Filepaths
CACHE_DIR = f"{codebase_path}/output/cache"
ENV_PATH = f"{codebase_path}/artifacts/.env"
MODELS_DIR = f"{codebase_path}/output/models"

✅ Codebase mounted at: /home/emmy/emmy/mlbio/hw4


In [3]:
# === IMPORT HF API KEY ===
from huggingface_hub import login

# Load HF token from artifacts/.env
env_path = ENV_PATH
if os.path.exists(env_path):
    with open(env_path, "r") as f:
        for line in f:
            if line.strip() and not line.startswith("#") and "=" in line:
                key, val = line.strip().split("=", 1)
                if key.strip() == "HF_TOKEN":
                    login(token=val.strip())
                    print("Successfully logged into Hugging Face Hub!")
                    break
else:
    print(f"Warning: {env_path} not found.")

Successfully logged into Hugging Face Hub!


In [4]:
# === CONFIGURATION ===
USE_UNSLOTH = False 
USE_DORA = True
MODEL_ID = "ibm-granite/granite-4.1-8b"
TRAIN_PATH_STRUCT = f"{CACHE_DIR}/train_structural.jsonl"
VAL_PATH_STRUCT   = f"{CACHE_DIR}/val_structural.jsonl"
TRAIN_PATH_FULL   = f"{CACHE_DIR}/train_full_info.jsonl"
VAL_PATH_FULL     = f"{CACHE_DIR}/val_full_info.jsonl"
STRUCTONLY_OUTPUT_DIR = f"{MODELS_DIR}/granite-4.1-8b_structOnly_DoRA"
FULLINFO_OUTPUT_DIR   = f"{MODELS_DIR}/granite-4.1-8b_fullInfo_DoRA"


In [5]:
# === IMPORT LIBRARIES ===
# IMPORTANT: Unsloth must be imported FIRST to apply all kernel patches
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig, get_peft_model
from src.utils.prompts import format_prompt

try:
    from unsloth import FastLanguageModel, is_bfloat16_supported
    USE_UNSLOTH = True
    print("✅ Unsloth available! Using Unsloth.")
except ImportError:
    USE_UNSLOTH = False
    print("⚠️ Unsloth NOT available! Falling back to standard Transformers + PEFT.")
    def is_bfloat16_supported():
        return torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False


⚠️ Unsloth NOT available! Falling back to standard Transformers + PEFT.


In [6]:
# === LOAD TOKENIZER & PREPARE SCHEMAS ===
# Load tokenizer from the base model
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

COMPUTE_DTYPE = torch.bfloat16 if is_bfloat16_supported() else torch.float16
print(f"Compute dtype: {COMPUTE_DTYPE}")

# Apply chat template formatting
import json as _json
def apply_chat_template(example, tokenizer):
    example = _json.loads(example["text"])
    messages = format_prompt(example)
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": prompt}


Compute dtype: torch.bfloat16


In [7]:
# === LoRA CONFIG ===
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_dora=USE_DORA,
)

In [8]:
# === MULTI-CONFIG SFT TRAINING LOOP ===
import gc
from transformers.trainer_utils import get_last_checkpoint
max_seq_length = 3600
configs = [
    # {
    #     "prompt_format": "fullInfo",
    #     "train_path": TRAIN_PATH_FULL,
    #     "val_path":   VAL_PATH_FULL,
    #     "output_dir": FULLINFO_OUTPUT_DIR
    # },
    
	{
        "prompt_format": "structOnly",
        "train_path": TRAIN_PATH_STRUCT,
        "val_path":   VAL_PATH_STRUCT,
        "output_dir": STRUCTONLY_OUTPUT_DIR
    }
]
use_bf16 = is_bfloat16_supported()
use_fp16 = not use_bf16
for config in configs:
    format_name = config["prompt_format"]
    print(f"\n================ STARTING TRAINING FOR {format_name} ==================")
    # 1. Load and process datasets
    print(f"Loading dataset from {config['train_path']}...")
    dataset = load_dataset("text", data_files={
        "train": config["train_path"],
        "val":   config["val_path"]
    })
    processed_dataset = dataset.map(lambda x: apply_chat_template(x, tokenizer))
    # 2 & 3. Load model and attach LoRA
    if USE_UNSLOTH:
        print(f"Loading {MODEL_ID} with Unsloth 4-bit...")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=MODEL_ID,
            max_seq_length=max_seq_length,
            dtype=COMPUTE_DTYPE,
            load_in_4bit=True,
        )
        model = FastLanguageModel.get_peft_model(
            model,
            r=8,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_alpha=16,
            lora_dropout=0,
            bias="none",
            use_dora=USE_DORA,
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )
    else:
        print(f"Loading {MODEL_ID} with Transformers/PEFT 4-bit...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=COMPUTE_DTYPE,
            bnb_4bit_use_double_quant=True
        )
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=COMPUTE_DTYPE,
        )
        model.gradient_checkpointing_enable()
        model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()
    # 4. SFT Configuration
    training_args = SFTConfig(
        output_dir=config["output_dir"],
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=1,
        gradient_checkpointing=True, 
        learning_rate=2e-4,
        logging_steps=10,
        logging_dir=f"{config['output_dir']}/logs",
        num_train_epochs=2,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        bf16=use_bf16,
        fp16=use_fp16,
        warmup_ratio=0.03,
        optim="paged_adamw_8bit",
        dataset_text_field="text",
        seed=42,
        # max_seq_length,
    )
    # 5. Initialize SFTTrainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=processed_dataset["train"],
        eval_dataset=processed_dataset["val"],
        processing_class=tokenizer,
        args=training_args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    # 6. Train (resume if checkpoint exists)
    last_checkpoint = get_last_checkpoint(config["output_dir"])
    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
        trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        trainer.train()
    trainer.save_model(config["output_dir"])
    # 7. Memory cleanup
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    print(f"🧹 Cleared GPU cache after {format_name}.")
    print("=================================================================\n")




================ STARTING TRAINING FOR structOnly ==================
Loading dataset from /home/emmy/emmy/mlbio/hw4/output/cache/train_structural.jsonl...


Loading ibm-granite/granite-4.1-8b with Transformers/PEFT 4-bit...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/363 [00:00<?, ?it/s]

/home/emmy/miniconda3/envs/mlbio_gpu/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable params: 26,337,280 || all params: 8,406,888,448 || trainable%: 0.3133


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
500,0.202370,0.204316,0.208537,3987916.000000,0.947734
1000,0.168637,0.176835,0.193828,7975966.000000,0.953955
1500,0.139050,0.154700,0.153532,11957784.000000,0.959441
2000,0.128349,0.135094,0.138573,15949615.000000,0.964883
2500,0.103474,0.118574,0.121677,19938588.000000,0.969671
2888,0.103158,0.110866,0.116870,23022300.000000,0.971880


🧹 Cleared GPU cache after structOnly.

